# 02 · LoRA vs PiSSA on SmolVLM-256M, TextVQA @ 1024 px (Part 1, step 3)

**Setup (same as the paper unless noted):** adapters on all LM linear layers, r = 16, `lora_alpha = r`, dropout 0, AdamW without weight decay, cosine schedule with 3% warmup.
Paper deviations: batch 8 (not 128), and fp32 weights with **fp16 autocast** (the T4 has no bf16).

**Data:** 2,000 train / 500 eval from the TextVQA *validation* split, disjoint by image. Training target = majority answer.

**Main grid:** {LoRA, PiSSA} × lr {2e-4, 2e-5 (paper value)} × seed {42, 43}, 300 steps each (~11 min on a T4), plus zero-shot on the eval split. ≈ 1.5 h total.
Same seed ⇒ same batches in the same order for LoRA and PiSSA.

Logs go to `MyDrive/en4554-a1/part1/runs/<run>/` and are flushed every step. Re-running a cell skips finished runs.

In [ ]:
# --- setup (run every session) ---
import os
REPO_URL = "https://github.com/KaveeshwaraBandara/en4554-lora-vlm-tool-agent.git"
REPO_DIR = "/content/en4554-lora-vlm-tool-agent"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt
from common import gpu_summary, get_out_dir
print(gpu_summary()); print("outputs ->", get_out_dir())

## 1 · Pipeline check (~3 min)
Check three things in the output:
1. the `[check]` line shows only the answer being supervised, e.g. `' some answer<end_of_utterance>\n'`;
2. no `SKIPPED` after the first couple of steps;
3. `sec/step`, which gives the time per run.

In [ ]:
!python -m part1_vlm_lora.src.train --method pissa --quick --force

## 1b · Diagnostic (~2 min)
Rebuilds the first training batch and checks where the high step-1 loss comes from: fp16, train mode, the adapter wrapper, or one long answer dominating the token-weighted mean. Paste the output back before starting the grid.

In [ ]:
!python -m part1_vlm_lora.src.diagnose

## 2 · Main grid
The first call streams the 2,800-example pool (~5 min, cached afterwards). If Colab disconnects, re-run this cell: finished runs are skipped.

In [ ]:
!bash part1_vlm_lora/run_grid.sh main

## 3 · Quick look

In [ ]:
from part1_vlm_lora.src.analysis import load_runs, summary_table, plot_curves
runs = load_runs(pattern="textvqa_*")
display(summary_table(runs).sort_values(["placement", "lr", "method", "seed"]))
fig = plot_curves(runs, placement="lm")

In [ ]:
# paste this back into the chat
print(summary_table(runs).sort_values(["placement", "lr", "method", "seed"]).to_csv(index=False))